# Task 0 [3 points]
For all tasks in this assignment, use the RecBole package (https://recbole.io/) and the Amazon
Beauty datasets. Get familiar with the package. For using the package for the entire pipeline (loading
and splitting the data, training the model, evaluation), you can obtain 3 points. These points may
be added to any of the following tasks.

# Task 1 [2 points]
1. Train and evaluate the SASRec model. Remember to split the data properly for the sequential
recommendations (last user interaction for testing, second last user interaction for validation).
Calculate the metrics (at least recall and nDCG) for k = 5, 10, 20 (note that you don’t need
to calculate the recommendations multiple times to do so).
2. Analyze the quality of recommendations for users with different sequence lengths:
- After performing the standard evaluation, divide the data based on the input sequence
length (e.g. 0 − 5, 5 − 10, 15 − 20, 20 − 30, 30+).
- Calculate how many sequences fall into each group.
- Calculate the metrics for each group.
- Visualize the results in a clear form. Describe your observations.

In [1]:
from collections import defaultdict
import math

import matplotlib.pyplot as plt
import numpy as np

import pandas as pd
import torch

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import get_model, get_trainer, init_seed

# np.float_ = np.float64
# np.complex_ = np.complex128
# np.unicode_ = np.str_

In [ ]:
task1_config_dict = {
    # Amazon Beauty sequential recommendation data
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "TIME_FIELD": "timestamp",
    "load_col": {"inter": ["user_id", "item_id", "timestamp"]},
    "ITEM_LIST_LENGTH_FIELD": "item_length",
    "LIST_SUFFIX": "_list",
    "MAX_ITEM_LIST_LENGTH": 50,

    # Sequential leave-one-out split: last item test, second last item validation
    "eval_args": {
        "group_by": "user",
        "order": "TO",
        "split": {"LS": "valid_and_test"},
        "mode": "full",
    },

    # Model and optimization
    "loss_type": "CE",
    "train_neg_sample_args": None,
    "epochs": 50,
    "stopping_step": 5,
    "train_batch_size": 2048,
    "eval_batch_size": 4096,
    "learning_rate": 1e-3,
    "hidden_size": 64,
    "inner_size": 256,
    "n_layers": 2,
    "n_heads": 2,
    "hidden_dropout_prob": 0.5,
    "attn_dropout_prob": 0.5,

    # Metrics
    "metrics": ["Recall", "NDCG"],
    "topk": [5, 10, 20],
    "valid_metric": "NDCG@10",
    "metric_decimal_place": 4,
    "seed": 42,
    "reproducibility": True,
}

config = Config(model="SASRec", dataset="amazon-beauty", config_dict=task1_config_dict)
init_seed(config["seed"], config["reproducibility"])

dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

model = get_model(config["model"])(config, train_data.dataset).to(config["device"])
trainer = get_trainer(config["MODEL_TYPE"], config["model"])(config, model)

best_valid_score, best_valid_result = trainer.fit(
    train_data,
    valid_data,
    saved=True,
    show_progress=True,
)
test_result = trainer.evaluate(
    test_data,
    load_best_model=True,
    show_progress=True,
)

print("Best validation score:", best_valid_score)
print("Best validation metrics:", best_valid_result)
print("Test metrics:", test_result)


/shared/programming/UWr/Sem8/ADM/List6/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
/shared/programming/UWr/Sem8/ADM/List6/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the inte

### Recommendation quality by input sequence length

For the subgroup analysis, the code below performs full-sort inference once with `k=20` and then slices the resulting ranks into `k = 5, 10, 20`. With leave-one-out testing each evaluated sequence has one relevant next item, so `Recall@k` is the fraction of sequences where the held-out item appears in the top-k list, and `NDCG@k` is `1 / log2(rank + 1)` when it appears.

In [ ]:
def sequence_length_bucket(length: int) -> str:
    lengths = [0, 5, 10, 15, 20, 30]

    for i in range(1, len(lengths)):
        if length <= lengths[i]:
            return f"{lengths[i - 1]}-{lengths[i]}"
    return f"{lengths[-1]}+"


def _as_numpy(x):
    return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)


def evaluate_by_sequence_length(model, eval_data, config, cutoffs=(5, 10, 20)):
    model.eval()
    max_k = max(cutoffs)
    length_field = config["ITEM_LIST_LENGTH_FIELD"]
    n_items = eval_data.dataset.item_num
    rows = []

    with torch.no_grad():
        for batched_data in eval_data:
            interaction, history_index, positive_u, positive_i = batched_data
            interaction = interaction.to(config["device"])

            scores = model.full_sort_predict(interaction).view(-1, n_items)
            scores[:, 0] = -torch.inf  # RecBole reserves item id 0 for padding.
            if history_index is not None:
                history_index = tuple(
                    x.to(config["device"]) if torch.is_tensor(x) else x for x in history_index
                )
                scores[history_index] = -torch.inf

            top_items = torch.topk(scores, k=max_k, dim=1).indices.cpu().numpy()
            lengths = _as_numpy(interaction[length_field]).astype(int)
            positive_u = _as_numpy(positive_u).astype(int)
            positive_i = _as_numpy(positive_i).astype(int)

            positives_by_row = defaultdict(set)
            for row_idx, item_idx in zip(positive_u, positive_i):
                positives_by_row[row_idx].add(item_idx)

            for row_idx, recommended in enumerate(top_items):
                relevant = positives_by_row.get(row_idx, set())
                if not relevant:
                    continue

                hit_rank = None
                for rank, item_idx in enumerate(recommended, start=1):
                    if item_idx in relevant:
                        hit_rank = rank
                        break

                row = {
                    "sequence_length": lengths[row_idx],
                    "bucket": sequence_length_bucket(lengths[row_idx]),
                }
                for k in cutoffs:
                    hit = hit_rank is not None and hit_rank <= k
                    row[f"Recall@{k}"] = float(hit)
                    row[f"NDCG@{k}"] = 1.0 / math.log2(hit_rank + 1) if hit else 0.0
                rows.append(row)

    per_sequence = pd.DataFrame(rows)
    bucket_order = ["0-5", "6-10", "11-15", "16-20", "21-30", "31+"]
    metric_cols = [f"{metric}@{k}" for k in cutoffs for metric in ("Recall", "NDCG")]
    grouped = (
        per_sequence.groupby("bucket", observed=False)
        .agg(sequences=("bucket", "size"), **{col: (col, "mean") for col in metric_cols})
        .reindex(bucket_order)
        .dropna(subset=["sequences"])
        .reset_index()
    )
    grouped["sequences"] = grouped["sequences"].astype(int)
    return per_sequence, grouped


per_sequence_metrics, length_group_metrics = evaluate_by_sequence_length(
    model,
    test_data,
    config,
    cutoffs=(5, 10, 20),
)

length_group_metrics


In [ ]:
display(length_group_metrics)

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, k in zip(axes, [5, 10, 20]):
    x = np.arange(len(length_group_metrics))
    width = 0.38
    ax.bar(x - width / 2, length_group_metrics[f"Recall@{k}"], width, label="Recall")
    ax.bar(x + width / 2, length_group_metrics[f"NDCG@{k}"], width, label="NDCG")
    ax.set_title(f"k = {k}")
    ax.set_xticks(x)
    ax.set_xticklabels(length_group_metrics["bucket"], rotation=30, ha="right")
    ax.set_xlabel("Input sequence length")
    ax.grid(axis="y", alpha=0.25)
axes[0].set_ylabel("Metric value")
axes[-1].legend(loc="upper right")
fig.suptitle("SASRec recommendation quality by sequence length")
fig.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(length_group_metrics["bucket"], length_group_metrics["sequences"], color="tab:gray")
plt.xlabel("Input sequence length")
plt.ylabel("Number of test sequences")
plt.title("Number of evaluated sequences per bucket")
plt.grid(axis="y", alpha=0.25)
plt.show()


# Task 2 [2 points]
1. Train and evaluate the LightGCN model. Use a proper data split. Calculate the metrics (at least
recall and nDCG) for k = 5, 10, 20 (note that you don’t need to calculate the recommendations
multiple times to do so).
2. Analyze the popularity of items:
- Calculate the popularity of each item. Make a bar plot with ordered popularities (from
largest to smallest).
- Analyze the popularity of the items you have recommended: how popular are the items
you recommend, and how often items of different popularity are recommended.
- By doing this analysis, you can observe the Matthew effect for Recommender Systems.
Explain it.

# Task 3 [3 points]
Use the LightGCN backbone and the Yelp2018 dataset. Compare several augmentation strategies
for contrastive learning in recommender systems.
1. Train models with the following augmentation strategies:
- Randomly drop 20% of edges in the graph (drop interactions).
- Randomly drop 20% of vertices in the graph (drop users/items).
- Normalize the embeddings to a unit sphere and add noise to the latent item representa-
tions.
Apply the augmentations separately in each mini-batch and in each epoch to ensure robustness.
2. Use the loss function: L = BPR+λ1(InfoNCE for users + InfoNCE for items)+λ2 regularization.
For drops, you can use the L2 regularization term; for noise, you can use the RBF function to
ensure uniformity.
3. For evaluation, repeat the analyzes from Task 1 and Task 2.
In this task, you are comparing SGL (https://dl.acm.org/doi/10.1145/3404835.3462862) and
SimGCL (https://dl.acm.org/doi/10.1145/3477495.3531937). You can find all the implementation
details in the literature. You can also use the implementations from the package
https://github.com/RUCAIBox/RecBole-GNN/tree/main